# 02 · Modeling & temporal evaluation

Baseline first, then LightGBM. The chronological split is part of the experimental design, not a convenience choice.

In [ ]:
from deposit_runoff.config import load_config
from deposit_runoff.demo_data import make_demo_daily_panel
from deposit_runoff.features import build_v1_dataset

cfg = load_config('../config/public.yaml')
d = cfg['data']
daily = make_demo_daily_panel(n_accounts=d['n_accounts'], start_date=str(d['start_date']), end_date=str(d['end_date']), seed=cfg['seed'])
snapshots = [*map(str, cfg['snapshots']['train']), str(cfg['snapshots']['validation']), str(cfg['snapshots']['test'])]
v1 = build_v1_dataset(daily, snapshots, **cfg['v1'])

In [ ]:
from deposit_runoff.modeling import prepare_model_frame, behavioral_baseline_score, evaluate_predictions, train_lightgbm
from deposit_runoff.split import chronological_split

v1 = prepare_model_frame(v1)
train, val, test = chronological_split(v1, train_dates=cfg['snapshots']['train'], validation_date=cfg['snapshots']['validation'], test_date=cfg['snapshots']['test'])
baseline = behavioral_baseline_score(val)
evaluate_predictions(val['runoff_flag'], baseline)

In [ ]:
model = train_lightgbm(train, val, cfg['model'])
test_pred = model.predict_proba(test[model.feature_name_])[:, 1]
evaluate_predictions(test['runoff_flag'], test_pred)

## Verified institutional result, sanitized for public display

The confidential analysis produced approximately **0.85 ROC-AUC**, **0.47 PR-AUC**, and **56% event capture in the highest-risk 10%** on an unseen monthly holdout. Synthetic-demo metrics are not expected to match those values.